In [ ]:
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt

In [ ]:
#Creation of initial sepsis and control split from sepsis3 list

sepsis3 = pd.read_csv("sepsis3-df.csv")
sepsis3 = sepsis3.rename(columns={'hadm_id':'HADM_ID'})

neonates = sepsis3[sepsis3['age'] == 0]
neonate_sepsis = neonates[neonates['sepsis-3'] == 1]
neonate_control = neonates[neonates['sepsis-3'] == 0]

control_ids_hadm = neonate_control['HADM_ID']
sepsis_ids_hadm = neonate_sepsis['HADM_ID']

In [ ]:
#Creation of auxillary datasets and subject_id to hadm_id translation

patients = pd.read_csv("/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4/PATIENTS.csv")
admissions = pd.read_csv("/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4/ADMISSIONS.csv")
mapping_ids = admissions[['SUBJECT_ID', 'HADM_ID']]

sepsis_admissions = admissions[admissions['HADM_ID'].isin(sepsis_ids_hadm)]
control_admissions = admissions[admissions['HADM_ID'].isin(control_ids_hadm)]

sepsis_ids = sepsis_admissions['SUBJECT_ID']
control_ids = control_admissions['SUBJECT_ID']

sepsis_patients = patients[patients['SUBJECT_ID'].isin(sepsis_ids)]
control_patients = patients[patients['SUBJECT_ID'].isin(control_ids)]

In [ ]:
#Collection of physiological data (control), filtered by select vitals

chunk_size = 5000000
chunk_vars = {}
vitals_map = {
    'heart_rate': [211, 220045],
    'respiratory_rate': [3603, 3337, 618, 220210],
    'rr_set':[619],
    'sp02': [646],
    'sa02': [834],
    'fio2': [190, 3420, 2981, 7570],
    'temp_c': [3655, 676, 677, 223762],
    'temp_f': [645, 678, 679, 223761],
    'temp_axillary_f':[3652],
    'temp_rectal_f':[3654, 6643],
    'o2_tcp': [3647, 3651],
    'bp_sys': [51, 3325, 455, 3313],
    'bp_dia': [8502, 8555],
    'map': [3324, 456, 3312, 52]
}

all_vital_ids = [idx for ids in vitals_map.values() for idx in ids]

reader = pd.read_csv(
    "/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4/CHARTEVENTS.csv", 
    chunksize=chunk_size,
    usecols=['SUBJECT_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM', 'VALUEUOM']
)

last_index = 0
for i, chunk in enumerate(reader):
    var_name = f'df_chunk_{i}'
    chunk = chunk[chunk['SUBJECT_ID'].isin(control_ids)]
    chunk = chunk[chunk['ITEMID'].isin(all_vital_ids)]
    id_to_name = {idx: name for name, ids in vitals_map.items() for idx in ids}
    chunk['LABEL'] = chunk['ITEMID'].map(id_to_name)
    chunk_vars[var_name] = chunk
    last_index = i
    print(f"\rProcessing: {var_name} | Rows in chunk: {len(chunk)}", end="", flush=True)

chunk_list = []
for key in list(chunk_vars.keys()):
    df = chunk_vars[key]
    if df.empty:
        continue
    chunk_list.append(df)

control_phys = pd.concat(chunk_list, ignore_index=True)
control_phys['CHARTTIME'] = pd.to_datetime(control_phys['CHARTTIME'])
control_phys = control_phys.sort_values(['SUBJECT_ID', 'CHARTTIME'])
    
total_rows = sum(len(c) for c in chunk_vars.values())
print(f"\n\n--- Execution Complete ---")
print(f"Total Chunks Created: {last_index + 1}")
print(f"Total Filtered Rows Stored: {total_rows:,}")
control_phys.head()

In [ ]:
#Collection of physiological data (sepsis)

chunk_vars_sepsis = {}

reader = pd.read_csv(
    "/oscar/data/shared/ursa/mimic-iii-clinical-database-1.4/CHARTEVENTS.csv", 
    chunksize=chunk_size,
    usecols=['SUBJECT_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM', 'VALUEUOM']
)

last_index = 0
for i, chunk in enumerate(reader):
    var_name = f'df_chunk_{i}'
    chunk = chunk[chunk['SUBJECT_ID'].isin(sepsis_ids)]
    chunk = chunk[chunk['ITEMID'].isin(all_vital_ids)]
    chunk['LABEL'] = chunk['ITEMID'].map(id_to_name)
    chunk_vars[var_name] = chunk
    last_index = i
    print(f"\rProcessing: {var_name} | Rows in chunk: {len(chunk)}", end="", flush=True)

chunk_list = []
for key in list(chunk_vars.keys()):
    df = chunk_vars[key]
    if df.empty:
        continue
    chunk_list.append(df)

sepsis_phys = pd.concat(chunk_list, ignore_index=True)
sepsis_phys['CHARTTIME'] = pd.to_datetime(sepsis_phys['CHARTTIME'])
sepsis_phys = sepsis_phys.sort_values(['SUBJECT_ID', 'CHARTTIME'])
    
total_rows = sum(len(c) for c in chunk_vars.values())
print(f"\n\n--- Execution Complete ---")
print(f"Total Chunks Created: {last_index + 1}")
print(f"Total Filtered Rows Stored: {total_rows:,}")
sepsis_phys.head()

In [ ]:
#Conducting density analysis (control)

def perform_density_audit(df):
    sparsity = df.groupby('LABEL').agg(
        total_records=('VALUENUM', 'count'),
        unique_patients=('SUBJECT_ID', 'nunique'),
        mean_val=('VALUENUM', 'mean'),
        std_val=('VALUENUM', 'std')
    )
    sparsity['avg_records_per_patient'] = sparsity['total_records'] / sparsity['unique_patients']
    
    patient_vital_counts = df.pivot_table(
        index='SUBJECT_ID', 
        columns='LABEL', 
        values='VALUENUM', 
        aggfunc='count'
    ).fillna(0)
    
    df = df.sort_values(['SUBJECT_ID', 'CHARTTIME'])
    df['time_diff'] = df.groupby(['SUBJECT_ID', 'LABEL'])['CHARTTIME'].diff().dt.total_seconds() / 60
    
    time_gaps = df.groupby('LABEL')['time_diff'].describe()[['50%', '75%', 'max']]
    time_gaps.columns = ['median_gap_mins', '75th_percentile_gap', 'max_gap']

    stay_durations = df.groupby('SUBJECT_ID')['CHARTTIME'].agg(['min', 'max'])
    stay_durations['total_hours'] = (stay_durations['max'] - stay_durations['min']).dt.total_seconds() / 3600
    
    return sparsity, patient_vital_counts, time_gaps, stay_durations


sparsity_stats, patient_completeness, gap_stats, stay_stats = perform_density_audit(control_phys)

print("\n### VARIABLE SPARSITY ###")
print(sparsity_stats)

print("\n### TYPICAL TIME GAPS (MINUTES) ###")
print(gap_stats)

print(f"\nAverage monitoring duration: {stay_stats['total_hours'].mean():.2f} hours")

In [ ]:
#Conducting density analysis (sepsis)

sparsity_stats, patient_completeness, gap_stats, stay_stats = perform_density_audit(sepsis_phys)

print("\n### VARIABLE SPARSITY ###")
print(sparsity_stats)

print("\n### TYPICAL TIME GAPS (MINUTES) ###")
print(gap_stats)

print(f"\nAverage monitoring duration: {stay_stats['total_hours'].mean():.2f} hours")

In [ ]:
#sepsis_phys.to_csv("sepsis_phys_new.csv", index=False)
#control_phys.to_csv("control_phys_new.csv", index=False)
sepsis_phys = pd.read_csv("sepsis_phys_new.csv")
control_phys = pd.read_csv("control_phys_new.csv")

In [ ]:
control_phys = control_phys.merge(mapping_ids[['SUBJECT_ID', 'HADM_ID']], on='SUBJECT_ID', how='left')
sepsis_phys = sepsis_phys.merge(mapping_ids[['SUBJECT_ID', 'HADM_ID']], on='SUBJECT_ID', how='left')

In [ ]:
def clean_and_standardize_vitals(df, df_admissions=None, mapping_ids=None):
    """
    Cleans and standardizes neonatal physiological data in long format.
    Assumes columns: ['SUBJECT_ID', 'CHARTTIME', 'LABEL', 'VALUENUM', 'VALUEUOM', 'ITEMID', 'HADM_ID']
    If df_admissions is provided, it pre-fills missing FiO2 records with 21.0 (room air) for the first 24 hours.
    """
    df = df.copy()
    
    # ---------------------------------------------------------
    # 1. TEMPERATURE CONVERSION & UNIFICATION
    # ---------------------------------------------------------
    is_f = df['LABEL'].isin(['temp_axillary_f', 'temp_rectal_f'])
    df.loc[is_f, 'VALUENUM'] = (df.loc[is_f, 'VALUENUM'] - 32) * (5.0 / 9.0)
    
    is_ax = df['LABEL'] == 'temp_axillary_f'
    df.loc[is_ax, 'VALUENUM'] += 0.5
    
    df['UNIFIED_LABEL'] = df['LABEL']
    temp_labels = ['temp_c', 'temp_axillary_f', 'temp_rectal_f']
    df.loc[df['LABEL'].isin(temp_labels), 'UNIFIED_LABEL'] = 'temperature'

    # ---------------------------------------------------------
    # 2. OUTLIER CLIPPING (Neonatal Ranges)
    # ---------------------------------------------------------
    bounds = {
        'heart_rate': (30, 250),
        'respiratory_rate': (10, 100),
        'sa02': (40, 100),
        'fio2': (20, 100),
        'map': (20, 100),
        'bp_sys': (30, 200),
        'bp_dia': (15, 170),
        'temp_c': (32, 42),          
        'temp_axillary_f': (32, 42), 
        'temp_rectal_f': (32, 42),
        'o2_tcp': (20, 100) 
    }

    for label, (lower, upper) in bounds.items():
        mask = df['LABEL'] == label
        df.loc[mask, 'VALUENUM'] = df.loc[mask, 'VALUENUM'].clip(lower=lower, upper=upper)

    # ---------------------------------------------------------
    # 3. BLOOD PRESSURE CALCULATION (MAP)
    # ---------------------------------------------------------
    bp_df = df[df['LABEL'].isin(['bp_sys', 'bp_dia', 'map'])]
    bp_pivot = bp_df.pivot_table(
        index=['SUBJECT_ID', 'CHARTTIME', 'HADM_ID'], 
        columns='LABEL', 
        values='VALUENUM'
    ).reset_index()

    for col in ['bp_sys', 'bp_dia', 'map']:
        if col not in bp_pivot.columns:
            bp_pivot[col] = np.nan

    missing_map_mask = bp_pivot['map'].isna() & bp_pivot['bp_sys'].notna() & bp_pivot['bp_dia'].notna()
    missing_maps = bp_pivot[missing_map_mask].copy()

    if not missing_maps.empty:
        missing_maps['VALUENUM'] = (missing_maps['bp_sys'] + 2 * missing_maps['bp_dia']) / 3.0
        missing_maps['LABEL'] = 'map_calculated' 
        missing_maps['UNIFIED_LABEL'] = 'map'
        
        # We don't have ITEMID/VALUEUOM for calculated map easily accessible from the pivot, 
        # but we can pass NaNs or specific values if needed.
        new_maps_df = missing_maps[['SUBJECT_ID', 'HADM_ID', 'CHARTTIME', 'LABEL', 'UNIFIED_LABEL', 'VALUENUM']]
        df = pd.concat([df, new_maps_df], ignore_index=True)

    # ---------------------------------------------------------
    # 4. FiO2 PRE-FILLING (Room Air = 21.0)
    # ---------------------------------------------------------
    if df_admissions is not None:
        df['CHARTTIME'] = pd.to_datetime(df['CHARTTIME'])
        df_admissions['intime'] = pd.to_datetime(df_admissions['intime'])
        
        # Round down admission time to the floor hour
        df_admissions['intime_floored'] = df_admissions['intime'].dt.floor('H')
        
        admissions_sub = df_admissions[['HADM_ID', 'intime_floored']].drop_duplicates()
        
        # Isolate existing FiO2 data and calculate its relative hour to floored admission
        fio2_df = df[df['UNIFIED_LABEL'] == 'fio2'].merge(admissions_sub, on='HADM_ID', how='inner')
        fio2_df['hour_bin'] = np.floor((fio2_df['CHARTTIME'] - fio2_df['intime_floored']).dt.total_seconds() / 3600.0)
        
        existing_pairs = fio2_df[['HADM_ID', 'hour_bin']].drop_duplicates()
        existing_pairs['exists'] = True
        
        subjects = admissions_sub['HADM_ID'].unique()
        hours = np.arange(28) # 0 through 27
        grid = pd.MultiIndex.from_product([subjects, hours], names=['HADM_ID', 'hour_bin']).to_frame(index=False)
        grid = grid.merge(admissions_sub, on='HADM_ID', how='inner')
        
        missing_grid = grid.merge(existing_pairs, on=['HADM_ID', 'hour_bin'], how='left')
        missing_grid = missing_grid[missing_grid['exists'].isnull()].copy()
        
        if not missing_grid.empty:
            missing_grid['CHARTTIME'] = missing_grid['intime_floored'] + pd.to_timedelta(missing_grid['hour_bin'], unit='h')
            missing_grid['LABEL'] = 'fio2_imputed'
            missing_grid['UNIFIED_LABEL'] = 'fio2'
            missing_grid['VALUENUM'] = 21.0
            
            # 4a. Fix NaNs for metadata
            missing_grid['ITEMID'] = 190.0
            missing_grid['VALUEUOM'] = '%'
            
            # 4b. Map SUBJECT_ID from HADM_ID
            if mapping_ids is not None:
                missing_grid = missing_grid.merge(mapping_ids[['HADM_ID', 'SUBJECT_ID']], on='HADM_ID', how='left')
            else:
                missing_grid['SUBJECT_ID'] = np.nan
                
            new_fio2_df = missing_grid[['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUENUM', 'VALUEUOM', 'LABEL', 'UNIFIED_LABEL']]
            df = pd.concat([df, new_fio2_df], ignore_index=True)

        # Append the floored intime to the main output dataframe for ALL rows
        df = df.merge(admissions_sub, on='HADM_ID', how='left')

    # ---------------------------------------------------------
    # 5. FINAL CLEANUP & CASTING
    # ---------------------------------------------------------
    df['CHARTTIME'] = df['CHARTTIME'].dt.floor('H')
    df = df.sort_values(by=['SUBJECT_ID', 'CHARTTIME']).reset_index(drop=True)
    
    # Cast IDs to Int64 (Pandas nullable integer type handles NaNs without breaking)
    for col in ['SUBJECT_ID', 'ITEMID', 'HADM_ID']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    return df

control_phys_clean = clean_and_standardize_vitals(control_phys, neonate_control, mapping_ids)
control_phys_clean

In [ ]:
sepsis_phys_clean = clean_and_standardize_vitals(sepsis_phys, neonate_sepsis, mapping_ids)
sepsis_phys_clean

In [ ]:
def verify_24h_continuous_data(df_vitals, max_allowed_gap_hours=4.0):
    """
    Verifies that patients have continuous data for core variables in the first 24 hours.
    
    Args:
        df_vitals: Long-format dataframe with [HADM_ID, CHARTTIME, UNIFIED_LABEL, VALUENUM]
        df_admissions: Dataframe with [HADM_ID, intime]
        max_allowed_gap_hours: Maximum acceptable gap between readings in hours.
    
    Returns:
        patient_summary: DataFrame with Pass/Fail status per HADM_ID.
        detailed_stats: DataFrame with gap and coverage stats per HADM_ID and Variable.
    """
    df = df_vitals.copy()
    # 1. Ensure datetime formats
    df['CHARTTIME'] = pd.to_datetime(df['CHARTTIME'])
    
    # 2. MATCH DATA TYPES: Cast HADM_ID to Int64 for safe merging
    if 'HADM_ID' in df_vitals.columns:
        df['HADM_ID'] = pd.to_numeric(df['HADM_ID'], errors='coerce').astype('Int64')
    
    # 4. Merge using the floored intime and calculate relative hour offsets
    df['offset_hours'] = (df['CHARTTIME'] - df['intime_floored']).dt.total_seconds() / 3600.0
    
    # 5. Filter to the core variables and the 0-24 hour window
    core_vars = ['heart_rate', 'sa02', 'fio2', 'respiratory_rate', 'temperature']
    df = df[df['UNIFIED_LABEL'].isin(core_vars)]
    
    # We include a small negative buffer (-1) in case vitals were charted just before the official 'intime'
    df_24h = df[(df['offset_hours'] >= -1.0) & (df['offset_hours'] <= 24.0)].copy()
    
    # 6. Sort to calculate accurate time gaps between consecutive readings
    df_24h = df_24h.sort_values(by=['HADM_ID', 'UNIFIED_LABEL', 'offset_hours'])
    
    # Calculate the time gap from the previous reading for the same patient and variable
    df_24h['time_gap'] = df_24h.groupby(['HADM_ID', 'UNIFIED_LABEL'])['offset_hours'].diff()
    
    # 7. Aggregate stats per Patient per Variable
    detailed_stats = df_24h.groupby(['HADM_ID', 'UNIFIED_LABEL']).agg(
        first_record_hr=('offset_hours', 'min'),
        last_record_hr=('offset_hours', 'max'),
        max_gap_hr=('time_gap', 'max'),
        total_records=('VALUENUM', 'count')
    ).reset_index()
    
    # Handle NaNs in max_gap (occurs if there's only 1 record)
    detailed_stats['max_gap_hr'] = detailed_stats['max_gap_hr'].fillna(26.0)
    
    # 8. Define the "Pass" Criteria for each variable
    detailed_stats['var_passed'] = (
        (detailed_stats['first_record_hr'] <= 2.0) &
        (detailed_stats['last_record_hr'] >= 22.0) &
        (detailed_stats['last_record_hr'] - detailed_stats['first_record_hr'] >= 22.0) &
        (detailed_stats['max_gap_hr'] <= max_allowed_gap_hours)
    )
    
    # 9. Roll up to the Patient Level
    patient_summary = detailed_stats.groupby('HADM_ID').agg(
        vars_present=('UNIFIED_LABEL', 'nunique'),
        vars_passed=('var_passed', 'sum')
    ).reset_index()
    
    # Patient passes if they have all 5 variables AND all 5 variables passed the gap/coverage check
    patient_summary['has_24h_continuous'] = (
        (patient_summary['vars_present'] == len(core_vars)) & 
        (patient_summary['vars_passed'] == len(core_vars))
    )
    valid_hids = patient_summary[patient_summary['has_24h_continuous']]['HADM_ID']
    df_valid_24h = df_24h[(df_24h['HADM_ID'].isin(valid_hids)) & 
                      (df_24h['offset_hours'] >= -1.0) & 
                      (df_24h['offset_hours'] <= 24.0)].copy()
    
    valid_stats = detailed_stats[detailed_stats['HADM_ID'].isin(valid_hids)]
    
    # ---------------------------------------------------------
    # 10. PRINT SUMMARY STATISTICS
    # ---------------------------------------------------------
    total_patients = len(patient_summary)
    passing_patients = patient_summary['has_24h_continuous'].sum()
    
    print("\n" + "="*40)
    print(" 24-HOUR CONTINUOUS DATA SUMMARY ")
    print("="*40)
    print(f"Total patients in initial cohort: {total_patients:,}")
    print(f"Patients passing all criteria:    {passing_patients:,}")
    
    if total_patients > 0:
        retention_rate = (passing_patients / total_patients) * 100
        print(f"Retention rate:                   {retention_rate:.2f}%")
        
        # Breakdown of why patients failed (optional but highly helpful for debugging)
        failed_patients = total_patients - passing_patients
        if failed_patients > 0:
            missing_vars = (patient_summary['vars_present'] < len(core_vars)).sum()
            gap_failures = failed_patients - missing_vars
            print(f"\nFailure Breakdown:")
            print(f"  - Missing core variables:       {missing_vars:,} patients")
            print(f"  - Unacceptable time gaps/span:  {gap_failures:,} patients")
    print("="*40 + "\n")
    
    return df_valid_24h.drop(columns=['offset_hours', 'time_gap']), valid_stats

In [ ]:
control_phys_full, control_phys_stats = verify_24h_continuous_data(control_phys_clean, max_allowed_gap_hours=4.0)
sepsis_phys_full, sepsis_phys_stats = verify_24h_continuous_data(sepsis_phys_clean, max_allowed_gap_hours=4.0)

In [ ]:
sepsis_phys_stats

In [ ]:
def build_dynamic_source_tensor(df_vitals, df_stats, cohort_label):
    core_vitals = ['heart_rate', 'temperature', 'respiratory_rate', 'fio2', 'sa02']

    source_labels = [
        'temp_c', 'temp_axillary_f', 'temp_rectal_f',
        'fio2', 'fio2_imputed',
        'heart_rate', 'respiratory_rate', 'sa02'
    ]

    channel_names = (
        core_vitals +
        [f'{v}_observed_mask' for v in core_vitals] +
        [f'source_{s}' for s in source_labels]
    )

    vital_to_idx = {v: i for i, v in enumerate(core_vitals)}
    source_to_idx = {s: i + (2 * len(core_vitals)) for i, s in enumerate(source_labels)}

    min_start_map = df_stats.groupby('HADM_ID')['first_record_hr'].min().to_dict()
    patient_ids = np.asarray(df_stats['HADM_ID'].unique(), dtype=np.int64)

    subject_lookup = (
        df_vitals[['HADM_ID', 'SUBJECT_ID']]
        .dropna()
        .drop_duplicates('HADM_ID')
        .assign(
            HADM_ID=lambda d: pd.to_numeric(d['HADM_ID']).astype('int64'),
            SUBJECT_ID=lambda d: pd.to_numeric(d['SUBJECT_ID']).astype('int64')
        )
        .set_index('HADM_ID')['SUBJECT_ID']
        .to_dict()
    )
    subject_ids = np.array([subject_lookup[int(hid)] for hid in patient_ids], dtype=np.int64)

    n_patients = len(patient_ids)
    n_channels = len(channel_names)

    X = np.full((n_patients, 24, n_channels), np.nan)
    y = np.full(n_patients, cohort_label)

    for i, hid in enumerate(patient_ids):
        p_data = df_vitals[df_vitals['HADM_ID'] == hid].copy()
        start_hr = min_start_map[hid]

        p_data['hour_bin'] = np.floor(
            (p_data['CHARTTIME'] - p_data['intime_floored']).dt.total_seconds() / 3600.0
        ).astype(int)

        p_data['tensor_idx'] = p_data['hour_bin'] - start_hr
        p_data = p_data[(p_data['tensor_idx'] >= 0) & (p_data['tensor_idx'] < 24)]

        grouped = p_data.groupby(['tensor_idx', 'UNIFIED_LABEL', 'LABEL'])

        for (t_idx, unified_lbl, raw_lbl), group in grouped:
            t_idx = int(t_idx)

            if unified_lbl in vital_to_idx:
                v_idx = vital_to_idx[unified_lbl]
                new_val = group['VALUENUM'].mean()

                if np.isnan(X[i, t_idx, v_idx]):
                    X[i, t_idx, v_idx] = new_val
                else:
                    X[i, t_idx, v_idx] = (X[i, t_idx, v_idx] + new_val) / 2.0

                X[i, t_idx, v_idx + len(core_vitals)] = 1.0

            if raw_lbl in source_to_idx:
                X[i, t_idx, source_to_idx[raw_lbl]] = 1.0

    X[:, :, len(core_vitals):] = np.nan_to_num(X[:, :, len(core_vitals):], nan=0.0)

    return X, y, patient_ids, subject_ids, channel_names

X_control, y_control, hadm_control, subject_control, channel_names = build_dynamic_source_tensor(
    control_phys_full, control_phys_stats, cohort_label=0
)

X_sepsis, y_sepsis, hadm_sepsis, subject_sepsis, _ = build_dynamic_source_tensor(
    sepsis_phys_full, sepsis_phys_stats, cohort_label=1
)

X_combined = np.concatenate([X_control, X_sepsis], axis=0)
y_combined = np.concatenate([y_control, y_sepsis], axis=0)
hadm_ids_combined = np.concatenate([hadm_control, hadm_sepsis], axis=0)
subject_ids_combined = np.concatenate([subject_control, subject_sepsis], axis=0)

In [ ]:
print(f"Tensor Shape: {X_combined.shape}")

value_nans = np.isnan(X_combined[:, :, 0:5]).sum()
print(f"Total NaNs in value channels: {value_nans}")

mask_nans = np.isnan(X_combined[:, :, 5:]).sum()
print(f"Total NaNs in mask/source channels: {mask_nans}")

unique_mask_vals = np.unique(X_combined[:, :, 5:])
print(f"Unique values in mask/source channels: {unique_mask_vals}")

# Heart-rate observed mask is channel 5, not 6.
first_step_masks = X_combined[:, 0, 5:10].sum(axis=1)
patients_with_empty_start = np.sum(first_step_masks == 0)
print(f"Patients with no observed core vital data at T=0: {patients_with_empty_start}")

In [ ]:
# source_temp_axillary_f is channel 11:
# 0-4 values, 5-9 observed masks, 10-17 source masks.
axillary_channel = channel_names.index('source_temp_axillary_f')
temp_value_channel = channel_names.index('temperature')
temp_mask_channel = channel_names.index('temperature_observed_mask')

axillary_indices = np.where(X_combined[:, :, axillary_channel] == 1)

if len(axillary_indices[0]) > 0:
    p_idx, t_idx = axillary_indices[0][0], axillary_indices[1][0]

    print(f"Sample Check - Patient {p_idx} at Hour {t_idx}:")
    print(f"  Axillary Source Mask: {X_combined[p_idx, t_idx, axillary_channel]}")
    print(f"  Temp Value:           {X_combined[p_idx, t_idx, temp_value_channel]}")
    print(f"  Temp Observed Mask:   {X_combined[p_idx, t_idx, temp_mask_channel]}")

In [ ]:
# Pick a random patient
sample_idx = 56 
patient_id = '199918'

# Plot Heart Rate (Channel 0) and its Mask (Channel 6)
plt.figure(figsize=(12, 4))
plt.step(range(24), X_combined[sample_idx, :, 0], where='post', label='Heart Rate (Value)')
plt.bar(range(24), X_combined[sample_idx, :, 6], alpha=0.3, label='Real Data Mask', color='orange')
plt.title(f"Tensor Check for HADM_ID: {patient_id}")
plt.xlabel("Tensor Index (Aligned to First Record)")
plt.legend()
plt.show()

In [ ]:
def interpolate_tensor_values(X_tensor, num_value_channels=5):
    """
    Applies linear interpolation to fill gaps in the value channels of a 3D tensor.
    Edges (Hour 0 or Hour 23) are filled using forward/backward fill.
    Mask channels are left completely untouched.
    """
    # Create a copy to prevent overwriting your original raw tensor in memory
    X_filled = np.copy(X_tensor)
    
    n_patients = X_filled.shape[0]
    
    for p in range(n_patients):
        for f in range(num_value_channels):
            # Extract the 24-hour sequence for this specific patient and feature
            series = pd.Series(X_filled[p, :, f])
            
            # interpolate(limit_direction='both') performs linear interpolation 
            # for internal gaps, AND handles forward/backward fill for the edges.
            series = series.interpolate(method='linear', limit_direction='both')
            
            # Fallback: If a patient somehow had 0 real readings for a feature 
            # (which your density filter should prevent), fill with 0 to avoid breaking the model.
            series = series.fillna(0)
            
            # Inject the filled data back into the tensor
            X_filled[p, :, f] = series.values
            
    return X_filled

# Execute on your combined tensor
X_interpolated = interpolate_tensor_values(X_combined, num_value_channels=5)

In [ ]:
# 1. Check Values (Should be 0 NaNs)
val_nans_before = np.isnan(X_combined[:, :, 0:5]).sum()
val_nans_after = np.isnan(X_interpolated[:, :, 0:5]).sum()

print(f"NaNs in Value Channels -> Before: {val_nans_before:,} | After: {val_nans_after}")

# 2. Check Masks (Should have no NaNs, and only contain 0.0 or 1.0)
unique_masks = np.unique(X_interpolated[:, :, 5:])
print(f"Unique values in Mask Channels: {unique_masks}")

In [ ]:
from sklearn.model_selection import train_test_split

def split_and_normalize_dataset(X, y, hadm_ids, subject_ids, test_size=0.10, val_size=0.10, num_vitals=5):
    X_train_val, X_test, y_train_val, y_test, hadm_train_val, hadm_test, subj_train_val, subj_test = train_test_split(
        X, y, hadm_ids, subject_ids,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    adj_val_size = val_size / (1 - test_size)

    X_train, X_val, y_train, y_val, hadm_train, hadm_val, subj_train, subj_val = train_test_split(
        X_train_val, y_train_val, hadm_train_val, subj_train_val,
        test_size=adj_val_size,
        random_state=42,
        stratify=y_train_val
    )

    def apply_normalization(tensor, stats):
        t_norm = np.copy(tensor)
        for f in range(num_vitals):
            t_norm[:, :, f] = (t_norm[:, :, f] - stats[f]['mean']) / stats[f]['std']
        return t_norm

    train_stats = {}
    for f in range(num_vitals):
        real_data = X_train[:, :, f][X_train[:, :, f + num_vitals] == 1.0]
        train_stats[f] = {
            'mean': np.mean(real_data),
            'std': np.std(real_data) if np.std(real_data) > 0 else 1e-8
        }

    X_train_norm = apply_normalization(X_train, train_stats)
    X_val_norm = apply_normalization(X_val, train_stats)
    X_test_norm = apply_normalization(X_test, train_stats)

    split_ids = {
        'train_hadm_ids': hadm_train,
        'val_hadm_ids': hadm_val,
        'test_hadm_ids': hadm_test,
        'train_subject_ids': subj_train,
        'val_subject_ids': subj_val,
        'test_subject_ids': subj_test,
    }

    return (X_train_norm, y_train), (X_val_norm, y_val), (X_test_norm, y_test), train_stats, split_ids

(X_train, y_train), (X_val, y_val), (X_test, y_test), final_stats, split_ids = split_and_normalize_dataset(
    X_interpolated,
    y_combined,
    hadm_ids_combined,
    subject_ids_combined,
    num_vitals=5
)

In [ ]:
norm_params = np.array([[final_stats[i]['mean'], final_stats[i]['std']] for i in range(len(final_stats))])

np.savez_compressed(
    'neonatal_sepsis_data_v1.npz',
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    norm_params=norm_params,
    channel_names=np.array(channel_names),
    label_names=np.array(['control', 'sepsis']),
    **split_ids
)

In [ ]:
control_ids = control_phys_full['SUBJECT_ID'].unique()
sepsis_ids = sepsis_phys_full['SUBJECT_ID'].unique()

control_patients = control_patients[control_patients['SUBJECT_ID'].isin(control_ids)]
control_admissions = control_admissions[control_admissions['SUBJECT_ID'].isin(control_ids)]

sepsis_patients = sepsis_patients[sepsis_patients['SUBJECT_ID'].isin(sepsis_ids)]
sepsis_admissions = sepsis_admissions[sepsis_admissions['SUBJECT_ID'].isin(sepsis_ids)]

print(control_ids.size)
print(sepsis_ids.size)

In [ ]:
gender_count = control_patients['GENDER'].value_counts(normalize=True)
gender_count

In [ ]:
gender_count = sepsis_patients['GENDER'].value_counts(normalize=True)
gender_count

In [ ]:
sepsis_demographics = sepsis_admissions['ETHNICITY'].value_counts()
sepsis_demographics

In [ ]:
control_demographics = control_admissions['ETHNICITY'].value_counts()
control_demographics

In [ ]:
def get_cohort_vitals_stats(df_cohort, cohort_name="Unnamed"):
    """
    Calculates Mean and Std Dev for a single cohort's physiological data.
    
    Args:
        df_cohort: The long-format dataframe (control_phys_full or sepsis_phys_full).
        cohort_name: String label for the printout.
    """
    # 1. Define variables of interest
    core_vars = ['heart_rate', 'temperature', 'respiratory_rate', 'fio2', 'sa02']
    
    # 2. Filter and calculate
    stats = (df_cohort[df_cohort['UNIFIED_LABEL'].isin(core_vars)]
             .groupby('UNIFIED_LABEL')['VALUENUM']
             .agg(['mean', 'std', 'min', 'max', 'count'])
             .reset_index())
    
    # Formatting for display
    stats.columns = ['Variable', 'Mean', 'StdDev', 'Min', 'Max', 'Count']
    
    print(f"\n" + "="*60)
    print(f" STATISTICAL SUMMARY: {cohort_name.upper()} COHORT ")
    print("="*60)
    print(stats.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    print("="*60 + "\n")
    
    return stats

# Usage:
control_stats_summary = get_cohort_vitals_stats(control_phys_full, "Control")
sepsis_stats_summary = get_cohort_vitals_stats(sepsis_phys_full, "Sepsis")